# Demo complete : que se passe-t-il quand l'utilisateur clique sur Lausanne ?

Ce notebook reproduit **etape par etape** tout ce que MeteoGlobe fait quand l'utilisateur clique sur la ville de Lausanne dans le globe 3D.

L'idee : tu executes chaque cellule avec Shift+Enter et tu vois exactement ce qui se passe a chaque etape, avec les vraies donnees recuperees en direct depuis Open-Meteo et Nominatim. C'est exactement ce que fait le backend `server.py` et le frontend `app.js`, mais ecrit en Python pur dans un seul notebook.

## Plan

1. Installation et imports
2. Le clic : on definit la coordonnee (Lausanne)
3. Etape 1 - Le backend appelle Open-Meteo (meteo actuelle)
4. Etape 2 - Le backend appelle Nominatim (nom du lieu)
5. Etape 3 - Le backend assemble la reponse `/api/weather`
6. Etape 4 - Le backend appelle Open-Meteo (previsions 7 jours)
7. Etape 5 - Le backend transforme les previsions
8. Etape 6 - Le frontend recoit le tout et choisit l'icone
9. Etape 7 - Recapitulatif : ce que l'utilisateur voit a l'ecran

## 1. Installation et imports

On a besoin de la librairie `requests` pour appeler les APIs externes. Si elle n'est pas deja installee, decommente la premiere ligne et execute la cellule.

In [1]:
# !pip install requests

import requests
import time
import json
from datetime import datetime, timezone

print("Imports OK")

Imports OK


## 2. Le clic : on definit la coordonnee

Quand l'utilisateur clique sur le globe, le frontend recupere la latitude et la longitude du point cliqué grace a Cesium. Pour Lausanne, on a :

In [2]:
lat = 46.5197
lon = 6.6323

print(f"Lieu cliqué : Lausanne")
print(f"Latitude  : {lat}")
print(f"Longitude : {lon}")
print()
print("Le frontend va maintenant envoyer deux requetes au backend :")
print(f"  1) GET /api/weather?lat={lat}&lon={lon}")
print(f"  2) GET /api/forecast?lat={lat}&lon={lon}")

Lieu cliqué : Lausanne
Latitude  : 46.5197
Longitude : 6.6323

Le frontend va maintenant envoyer deux requetes au backend :
  1) GET /api/weather?lat=46.5197&lon=6.6323
  2) GET /api/forecast?lat=46.5197&lon=6.6323


## 3. Etape 1 : le backend appelle Open-Meteo (meteo actuelle)

Le backend recoit la requete `/api/weather` et fait un appel HTTP vers Open-Meteo pour recuperer la meteo actuelle. C'est exactement ce que fait la fonction `weather_payload()` dans `server.py`.

Le parametre `current` envoye a Open-Meteo contient une liste de **4 champs** separes par des virgules. On ne demande que ce qui est reellement affiche dans l'interface, pas plus.

| Nom du champ | Ce que ca donne en retour |
|---|---|
| `temperature_2m` | Temperature de l'air a 2 metres du sol, en °C |
| `apparent_temperature` | Temperature ressentie (prend en compte vent et humidite), en °C |
| `weather_code` | Code WMO de la condition meteo (0, 1, 2, 45, 61, 95, etc.) |
| `is_day` | Vaut `1` s'il fait jour a cet endroit, `0` sinon |

On utilise aussi `timezone=auto` pour qu'Open-Meteo nous renvoie le decalage UTC du lieu (`utc_offset_seconds`), ce qui permet ensuite d'afficher correctement l'heure locale dans le panneau.

In [ ]:
url = "https://api.open-meteo.com/v1/forecast"
params = {
    "latitude": lat,
    "longitude": lon,
    "current": "temperature_2m,apparent_temperature,weather_code,is_day",
    "timezone": "auto",
}

response = requests.get(url, params=params, timeout=10)
open_meteo_current = response.json()

print("Reponse brute d'Open-Meteo (extrait 'current') :")
print(json.dumps(open_meteo_current.get("current", {}), indent=2))
print()
print(f"utc_offset_seconds : {open_meteo_current.get('utc_offset_seconds')}")

On voit la temperature, la temperature ressentie, le code WMO de la meteo actuelle, et `is_day` qui vaut 1 ou 0 selon qu'il fait jour ou nuit a Lausanne. Le champ `utc_offset_seconds` au-dessous donne le decalage UTC du lieu en secondes (par exemple 7200 = UTC+2 en heure d'ete pour la Suisse).

## 4. Etape 2 : le backend appelle Nominatim (nom du lieu)

Open-Meteo ne donne pas le nom du lieu, juste les donnees meteo. Le backend appelle donc **Nominatim** (un service d'OpenStreetMap) pour faire un *reverse geocoding* : convertir une coordonnee en nom de ville.

Note : Nominatim limite a 1 requete par seconde. Le vrai backend a un mutex pour respecter ca, ici on s'en occupe pas parce qu'on fait une seule requete.

In [4]:
url = "https://nominatim.openstreetmap.org/reverse"
params = {
    "lat": lat,
    "lon": lon,
    "format": "jsonv2",
    "addressdetails": 1,
    "zoom": 14,
}
headers = {
    "User-Agent": "MeteoGlobe-Demo/1.0",
    "Accept-Language": "fr",
}

response = requests.get(url, params=params, headers=headers, timeout=10)
nominatim_data = response.json()

print("Reponse brute de Nominatim :")
print(json.dumps(nominatim_data, indent=2, ensure_ascii=False))

Reponse brute de Nominatim :
{
  "place_id": 83502999,
  "licence": "Data © OpenStreetMap contributors, ODbL 1.0. http://osm.org/copyright",
  "osm_type": "relation",
  "osm_id": 1685018,
  "lat": "46.5218269",
  "lon": "6.6327025",
  "category": "boundary",
  "type": "administrative",
  "place_rank": 16,
  "importance": 0.7044362162942157,
  "addresstype": "city",
  "name": "Lausanne",
  "display_name": "Lausanne, District de Lausanne, Vaud, Suisse",
  "address": {
    "city": "Lausanne",
    "county": "District de Lausanne",
    "state": "Vaud",
    "ISO3166-2-lvl4": "CH-VD",
    "country": "Suisse",
    "country_code": "ch"
  },
  "boundingbox": [
    "46.4548726",
    "46.6025814",
    "6.5838630",
    "6.7208093"
  ]
}


In [5]:
# Le backend extrait juste le nom de la ville et le code pays
address = nominatim_data.get("address", {})
place_name = (
    address.get("city")
    or address.get("town")
    or address.get("village")
    or address.get("municipality")
    or "Selected location"
)
country_code = address.get("country_code", "").upper()

print(f"Nom du lieu : {place_name}")
print(f"Pays        : {country_code}")

Nom du lieu : Lausanne
Pays        : CH


## 5. Etape 3 : le backend assemble la reponse `/api/weather`

Le backend prend les donnees brutes d'Open-Meteo + le nom de Nominatim et construit un objet JSON propre, pret a etre consomme par le frontend. C'est ce que fait la fonction `weather_payload()`.

In [ ]:
current = open_meteo_current["current"]

# current.time est en heure locale (timezone=auto). On recupere le decalage
# UTC du lieu pour le convertir en timestamp Unix UTC.
utc_offset = open_meteo_current.get("utc_offset_seconds", 0)
local_unix = int(datetime.fromisoformat(current["time"]).replace(tzinfo=timezone.utc).timestamp())
dt_unix = local_unix - utc_offset

# is_day = 1 -> on simule sunrise dans le passe et sunset dans le futur (et inverse pour la nuit)
is_day = current["is_day"]
sunrise = dt_unix - 6 * 3600 if is_day == 1 else dt_unix + 6 * 3600
sunset  = dt_unix + 6 * 3600 if is_day == 1 else dt_unix - 6 * 3600

weather_response = {
    "_source": "open-meteo-current",
    "coord": {"lat": lat, "lon": lon},
    "weather": [{"id": current["weather_code"], "description": "weather"}],
    "main": {
        "temp": current["temperature_2m"],
        "feels_like": current["apparent_temperature"],
    },
    "sys": {"country": country_code, "sunrise": sunrise, "sunset": sunset},
    "dt": dt_unix,
    "timezone": utc_offset,
    "name": place_name,
}

print("Reponse JSON envoyee au frontend par /api/weather :")
print(json.dumps(weather_response, indent=2, ensure_ascii=False))

## 6. Etape 4 : le backend appelle Open-Meteo (previsions 7 jours)

Pendant ce temps, le frontend a aussi demande `/api/forecast`. Le backend fait un deuxieme appel a Open-Meteo, cette fois avec des parametres differents pour avoir les previsions horaires et journalieres.

In [7]:
url = "https://api.open-meteo.com/v1/forecast"
params = {
    "latitude": lat,
    "longitude": lon,
    "hourly": "temperature_2m,precipitation_probability,precipitation,weather_code",
    "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum,weather_code",
    "forecast_days": 7,
    "timezone": "auto",
}

response = requests.get(url, params=params, timeout=10)
open_meteo_forecast = response.json()

print("Cles disponibles dans la reponse :", list(open_meteo_forecast.keys()))
print()
print("Premieres heures de previsions horaires :")
hourly = open_meteo_forecast["hourly"]
for i in range(3):
    print(f"  {hourly['time'][i]} -> {hourly['temperature_2m'][i]}°C, code WMO {hourly['weather_code'][i]}, pluie {hourly['precipitation'][i]}mm")

Cles disponibles dans la reponse : ['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'hourly_units', 'hourly', 'daily_units', 'daily']

Premieres heures de previsions horaires :
  2026-04-08T00:00 -> 14.7°C, code WMO 0, pluie 0.0mm
  2026-04-08T01:00 -> 14.5°C, code WMO 0, pluie 0.0mm
  2026-04-08T02:00 -> 13.8°C, code WMO 0, pluie 0.0mm


## 7. Etape 5 : le backend transforme les previsions

Open-Meteo renvoie des **tableaux paralleles** (un tableau pour les heures, un pour les temperatures, un pour les codes meteo, etc.). C'est verbeux et pas pratique a manipuler. Le backend transforme ca en deux listes propres : `hourly` (les 48 prochaines heures pour le graphique) et `daily` (les 7 prochains jours pour les cartes de prevision).

In [8]:
# Transformation hourly -> liste de dicts
hourly_data = []
for i in range(min(48, len(hourly["time"]))):
    hourly_data.append({
        "time": hourly["time"][i],
        "temp": hourly["temperature_2m"][i],
        "precip": hourly["precipitation"][i],
        "pop": hourly["precipitation_probability"][i],
    })

# Transformation daily -> liste de dicts
daily_raw = open_meteo_forecast["daily"]
daily_data = []
for i in range(len(daily_raw["time"])):
    daily_data.append({
        "date": daily_raw["time"][i],
        "temp_max": daily_raw["temperature_2m_max"][i],
        "temp_min": daily_raw["temperature_2m_min"][i],
        "precip": daily_raw["precipitation_sum"][i],
        "code": daily_raw["weather_code"][i],
    })

forecast_response = {
    "hourly": hourly_data,
    "daily": daily_data,
    "_source": "open-meteo",
}

print(f"Nombre d'heures dans hourly : {len(hourly_data)}")
print(f"Nombre de jours dans daily  : {len(daily_data)}")
print()
print("Previsions des 7 prochains jours :")
for d in daily_data:
    print(f"  {d['date']} : min {d['temp_min']}°C / max {d['temp_max']}°C, pluie {d['precip']}mm, code {d['code']}")

Nombre d'heures dans hourly : 48
Nombre de jours dans daily  : 7

Previsions des 7 prochains jours :
  2026-04-08 : min 11.5°C / max 22.5°C, pluie 0.0mm, code 0
  2026-04-09 : min 12.2°C / max 22.5°C, pluie 0.0mm, code 1
  2026-04-10 : min 12.5°C / max 20.2°C, pluie 0.0mm, code 3
  2026-04-11 : min 11.7°C / max 21.8°C, pluie 0.0mm, code 3
  2026-04-12 : min 8.8°C / max 15.1°C, pluie 3.6mm, code 61
  2026-04-13 : min 6.2°C / max 8.7°C, pluie 3.3mm, code 61
  2026-04-14 : min 6.3°C / max 8.1°C, pluie 4.5mm, code 61


## 8. Etape 6 : le frontend recoit le tout et choisit l'icone

Le frontend a maintenant les deux reponses (`weather_response` et `forecast_response`). Il doit afficher une icone meteo dans le panneau lateral. Pour ca, il utilise la table `WMO_TO_METEO` et la fonction `getMeteoIcon()`.

In [9]:
WMO_TO_METEO = {
    0:  {"d": 1,  "n": 101},  1:  {"d": 2,  "n": 102},  2:  {"d": 3,  "n": 103},
    3:  {"d": 5,  "n": 105},  45: {"d": 25, "n": 25},   48: {"d": 25, "n": 25},
    51: {"d": 7,  "n": 7},    53: {"d": 7,  "n": 7},    55: {"d": 7,  "n": 7},
    56: {"d": 26, "n": 26},   57: {"d": 26, "n": 26},
    61: {"d": 7,  "n": 7},    63: {"d": 8,  "n": 8},    65: {"d": 9,  "n": 9},
    66: {"d": 26, "n": 26},   67: {"d": 26, "n": 26},
    71: {"d": 13, "n": 13},   73: {"d": 14, "n": 14},   75: {"d": 15, "n": 15},
    77: {"d": 15, "n": 15},
    80: {"d": 17, "n": 17},   81: {"d": 18, "n": 18},   82: {"d": 19, "n": 19},
    85: {"d": 17, "n": 17},   86: {"d": 15, "n": 15},
    95: {"d": 23, "n": 23},   96: {"d": 20, "n": 20},   99: {"d": 21, "n": 21},
}

def is_daytime(data):
    """Vrai si le timestamp data['dt'] est entre sunrise et sunset."""
    if not data or "sys" not in data:
        return True
    sys = data["sys"]
    if not sys.get("sunrise") or not sys.get("sunset"):
        return True
    return sys["sunrise"] <= data["dt"] <= sys["sunset"]

def get_meteo_icon(wmo_code, daytime=True):
    """Convertit un code WMO en numero d'icone MeteoSwiss."""
    m = WMO_TO_METEO.get(wmo_code)
    if m is not None:
        return m["d"] if daytime else m["n"]
    return 1 if daytime else 101

print("Fonctions definies.")

Fonctions definies.


In [10]:
# Le frontend regarde s'il fait jour ou nuit a Lausanne en ce moment
jour = is_daytime(weather_response)

# Le frontend extrait le code WMO
code_wmo = weather_response["weather"][0]["id"]

# Le frontend choisit l'icone correspondante
icone = get_meteo_icon(code_wmo, daytime=jour)

print(f"Code WMO recu       : {code_wmo}")
print(f"Il fait jour        : {jour}")
print(f"Numero d'icone      : {icone}")
print(f"URL chargee         : /api/icon/{icone}")
print(f"Fichier servi       : public/icons/{icone}.svg")

Code WMO recu       : 0
Il fait jour        : True
Numero d'icone      : 1
URL chargee         : /api/icon/1
Fichier servi       : public/icons/1.svg


## 9. Recapitulatif : ce que l'utilisateur voit a l'ecran

Le frontend a tout ce qu'il faut pour remplir le panneau lateral. Voici ce qui s'affiche dans le panneau de droite :

In [ ]:
temp = weather_response["main"]["temp"]
feels = weather_response["main"]["feels_like"]

print("=" * 50)
print(f"  PANNEAU METEO")
print("=" * 50)
print(f"  Lieu        : {weather_response['name']} ({weather_response['sys']['country']})")
print(f"  Coordonnees : {lat}, {lon}")
print(f"  Heure locale: {datetime.fromtimestamp(weather_response['dt'] + weather_response['timezone'], tz=timezone.utc).strftime('%H:%M')}")
print("-" * 50)
print(f"  Icone       : {icone}.svg ({'jour' if jour else 'nuit'})")
print(f"  Temperature : {temp}°C  (ressenti {feels}°C)")
print("-" * 50)
print(f"  Previsions sur 7 jours :")
for d in forecast_response["daily"]:
    icone_jour = get_meteo_icon(d["code"], daytime=True)
    print(f"    {d['date']} : {d['temp_min']:>5.1f}°C / {d['temp_max']:>5.1f}°C  pluie {d['precip']:>4.1f}mm  icone {icone_jour}")
print("=" * 50)

## A retenir

Pour un seul clic sur Lausanne, voici tout ce qui s'est passe :

1. **Le frontend** a envoye 2 requetes au backend (`/api/weather` et `/api/forecast`).
2. **Le backend** a fait 3 appels a des APIs externes :
   - 1 a Open-Meteo pour la meteo actuelle
   - 1 a Nominatim pour le nom de la ville
   - 1 a Open-Meteo pour les previsions
3. **Le backend** a transforme les reponses verbeuses d'Open-Meteo en JSON simples.
4. **Le frontend** a utilise la table `WMO_TO_METEO` pour choisir l'icone et a affiche le panneau.

Le tout se fait en quelques centaines de millisecondes la premiere fois, et **quasi instantanement** les fois suivantes grace au cache du backend (15 min pour la meteo actuelle, 30 min pour les previsions, 30 jours pour les noms de Nominatim).

## Pour aller plus loin

Tu peux modifier `lat` et `lon` au debut du notebook pour tester avec n'importe quelle autre ville :

- Paris : `lat = 48.8566, lon = 2.3522`
- Tokyo : `lat = 35.6762, lon = 139.6503`
- Sydney : `lat = -33.8688, lon = 151.2093`
- Reykjavik : `lat = 64.1466, lon = -21.9426`

Et re-executer toutes les cellules. Tu verras la meteo en direct, le nom de la ville, et les previsions sur 7 jours pour cet endroit precis.